# Step 1 — 신문사 탐색 (Newspaper Discovery) v4
**확인된 구조**: `data['content.results']`, 25개씩 페이지네이션,
`location_state={'value':'...'}`, `number_lccn=[...]`, `number_first_issue/last_issue={'label':'YYYY-MM-DD'}`

**변경**: 주별로 `fa=location_state:<주>` 서버 필터 + `c=100`으로 요청, 1907/1909/1911 커버 여부로 필터링.

## Cell 1 — 환경 설정

In [1]:
!pip install -q ftfy
import requests, xml.etree.ElementTree as ET
import pandas as pd, time, re
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import ftfy
from google.colab import drive
drive.mount('/content/drive')

def make_session():
    s=requests.Session()
    retry=Retry(total=2,connect=2,read=2,backoff_factor=0.5,
                status_forcelist=[429,500,502,503,504],
                allowed_methods=['GET'],raise_on_status=False)
    s.mount('https://',HTTPAdapter(max_retries=retry))
    s.mount('http://', HTTPAdapter(max_retries=retry))
    s.headers.update({'User-Agent':'KNU-causal-inference/1.0'})
    return s

SESSION=make_session()
REQUEST_TIMEOUT=(8,25)
RATE_LIMIT=0.3
CA_BASE='https://chroniclingamerica.loc.gov'
print('완료')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00
Mounted at /content/drive
완료


## Cell 2 — 주별 서버 필터로 신문 목록 조회
`fa=location_state:<주>` 필터 + `c=100`(페이지당 100개) + 페이지네이션(`sp=1,2,3`).
각 신문의 발행기간(`number_first_issue`~`number_last_issue`)이 1907~1911을 포함하는지 확인.

In [2]:
TARGET_STATES = ['west virginia','illinois','new york','pennsylvania','ohio','massachusetts']
NEEDED_YEARS = (1907, 1911)  # 이 범위를 모두 포함해야 함
MAX_PAGES = 3                # 주당 최대 3페이지(최대 300개) 조회

def year_from_label(label):
    """'1847-03-03' -> 1847"""
    m = re.match(r'(\d{4})', str(label))
    return int(m.group(1)) if m else None

def fetch_state_newspapers(state):
    all_items = []
    for sp in range(1, MAX_PAGES+1):
        params = {
            'fo': 'json', 'c': 100, 'sp': sp,
            'fa': f'location_state:{state}',
        }
        try:
            r = SESSION.get(f'{CA_BASE}/newspapers.json', params=params, timeout=REQUEST_TIMEOUT)
            if r.status_code != 200:
                break
            data = r.json()
            items = data.get('content.results', [])
            if not items:
                break
            all_items.extend(items)
            time.sleep(RATE_LIMIT)
            if len(items) < 100:
                break  # 마지막 페이지
        except Exception as e:
            print(f'    오류({state}, page{sp}): {type(e).__name__}')
            break
    return all_items

candidates = []
for state in TARGET_STATES:
    items = fetch_state_newspapers(state)
    # 필터 작동 여부 확인: location_state.value가 실제로 우리가 요청한 주인지
    matched_state_items = [
        it for it in items
        if isinstance(it.get('location_state'), dict)
        and it['location_state'].get('value','').lower() == state
    ]
    kept = 0
    for it in matched_state_items:
        lccn_list = it.get('number_lccn', [])
        lccn = lccn_list[0] if lccn_list else ''
        title = it.get('title','')
        first = it.get('number_first_issue',{})
        last  = it.get('number_last_issue',{})
        y1 = year_from_label(first.get('label')) if isinstance(first,dict) else None
        y2 = year_from_label(last.get('label'))  if isinstance(last,dict)  else None
        if lccn and y1 is not None and y2 is not None \
           and y1 <= NEEDED_YEARS[0] and y2 >= NEEDED_YEARS[1]:
            candidates.append({
                'lccn': lccn, 'name': title,
                'state': state.title(),
                'first_year': y1, 'last_year': y2,
            })
            kept += 1
    print(f'  {state.title():15s}: fa필터 응답 {len(items)}개 '
          f'(주 일치 {len(matched_state_items)}개) → 1907-1911 커버 {kept}개')

cand_df = pd.DataFrame(candidates, columns=['lccn','name','state','first_year','last_year']).drop_duplicates(subset='lccn')
print(f'\n총 후보 신문사: {len(cand_df)}개')
if len(cand_df) > 0:
    print(cand_df[['name','state','first_year','last_year']].to_string(index=False))
else:
    print('[경고] 후보 0개 — fa 필터가 작동하지 않는 경우, 다음 셀에서 필터 없이 재시도합니다.')

  West Virginia  : fa필터 응답 0개 (주 일치 0개) → 1907-1911 커버 0개
  Illinois       : fa필터 응답 0개 (주 일치 0개) → 1907-1911 커버 0개
  New York       : fa필터 응답 0개 (주 일치 0개) → 1907-1911 커버 0개
  Pennsylvania   : fa필터 응답 0개 (주 일치 0개) → 1907-1911 커버 0개
  Ohio           : fa필터 응답 0개 (주 일치 0개) → 1907-1911 커버 0개
  Massachusetts  : fa필터 응답 0개 (주 일치 0개) → 1907-1911 커버 0개

총 후보 신문사: 0개
[경고] 후보 0개 — fa 필터가 작동하지 않는 경우, 다음 셀에서 필터 없이 재시도합니다.


## Cell 2b — (대비) fa 필터 실패 시 — 필터 없이 페이지네이션 + 클라이언트 필터링
Cell 2에서 후보가 0개였다면 이 셀을 실행. 전체 목록을 더 많이 가져와서 직접 필터.

In [3]:
if len(cand_df) == 0:
    print('필터 없이 전체 목록 조회 (최대 20페이지 = 2000개)...')
    all_items = []
    for sp in range(1, 21):
        params = {'fo':'json','c':100,'sp':sp}
        try:
            r = SESSION.get(f'{CA_BASE}/newspapers.json', params=params, timeout=REQUEST_TIMEOUT)
            if r.status_code != 200: break
            items = r.json().get('content.results', [])
            if not items: break
            all_items.extend(items)
            time.sleep(RATE_LIMIT)
            if len(items) < 100: break
        except Exception as e:
            print(f'  오류(page{sp}): {type(e).__name__}'); break
    print(f'총 {len(all_items)}개 수집')

    candidates = []
    target_set = set(TARGET_STATES)
    for it in all_items:
        ls = it.get('location_state')
        state_val = ls.get('value','').lower() if isinstance(ls,dict) else ''
        if state_val not in target_set: continue
        lccn_list = it.get('number_lccn', [])
        lccn = lccn_list[0] if lccn_list else ''
        first = it.get('number_first_issue',{})
        last  = it.get('number_last_issue',{})
        y1 = year_from_label(first.get('label')) if isinstance(first,dict) else None
        y2 = year_from_label(last.get('label'))  if isinstance(last,dict)  else None
        if lccn and y1 is not None and y2 is not None \
           and y1 <= NEEDED_YEARS[0] and y2 >= NEEDED_YEARS[1]:
            candidates.append({'lccn':lccn,'name':it.get('title',''),
                                'state':state_val.title(),'first_year':y1,'last_year':y2})

    cand_df = pd.DataFrame(candidates, columns=['lccn','name','state','first_year','last_year']).drop_duplicates(subset='lccn')
    print(f'\n총 후보 신문사: {len(cand_df)}개')
    if len(cand_df) > 0:
        print(cand_df.groupby('state').size().to_string())
else:
    print('Cell 2에서 이미 후보를 찾았으므로 스킵.')

필터 없이 전체 목록 조회 (최대 20페이지 = 2000개)...
총 0개 수집

총 후보 신문사: 0개


## Cell 3 — 신문사당 5개 날짜 빠른 테스트

In [4]:
MAX_PER_STATE = 8

KNOWN_GOOD = [
    {'lccn':'sn83030272','name':'The Sun','state':'New York'},
    {'lccn':'sn83045555','name':'Deseret Evening News','state':'Utah'},
    {'lccn':'sn84020645','name':'Montgomery Advertiser','state':'Alabama'},
    {'lccn':'sn85042462','name':'Los Angeles Herald','state':'California'},
    {'lccn':'sn85058130','name':'Salt Lake Herald','state':'Utah'},
    {'lccn':'sn83045604','name':'Washington Evening Star','state':'D.C.'},
]

if len(cand_df) > 0:
    limited = (cand_df.groupby('state', group_keys=False)
                       .apply(lambda g: g.head(MAX_PER_STATE)))
    new_candidates = limited[['lccn','name','state']].to_dict('records')
else:
    new_candidates = []

all_candidates = KNOWN_GOOD + new_candidates
seen_lccn = set()
dedup = []
for c in all_candidates:
    if c['lccn'] not in seen_lccn:
        seen_lccn.add(c['lccn'])
        dedup.append(c)
all_candidates = dedup
print(f'테스트할 신문사 총: {len(all_candidates)}개')

TEST_DATES = [
    ('1907-11-09', 'monongah_pre'),
    ('1907-12-07', 'monongah_post'),
    ('1909-11-14', 'cherry_post'),
    ('1911-03-03', 'triangle_pre'),
    ('1911-03-26', 'triangle_post'),
]

def extract_text(content_bytes):
    try:
        root = ET.fromstring(content_bytes)
    except: return ''
    ns = root.tag.split('}')[0].strip('{') if '}' in root.tag else ''
    P = f'{{{ns}}}' if ns else ''
    words=[s.get('CONTENT','') for s in root.iter(f'{P}String') if s.get('CONTENT','').strip()]
    text=' '.join(words)
    return ftfy.fix_text(text)

def ocr_quality(text):
    words=text.split()
    if not words: return 0
    lw=sum(1 for w in words if len(w)>=2)
    al=sum(1 for c in text if c.isalpha())
    return round((lw/len(words))*0.5+(al/max(len(text),1))*0.5,3)

def test_url(lccn, date_str, seq=1):
    url = f'{CA_BASE}/lccn/{lccn}/{date_str}/ed-1/seq-{seq}/ocr.xml'
    try:
        r = SESSION.get(url, timeout=REQUEST_TIMEOUT, allow_redirects=True)
        if r.status_code==200 and b'<alto' in r.content[:1000].lower():
            text = extract_text(r.content)
            return True, len(text.split()), ocr_quality(text)
        return False, 0, 0
    except Exception:
        return False, 0, 0

print(f'\n신문사당 {len(TEST_DATES)}개 날짜 테스트 시작...\n')
results = []
for c in all_candidates:
    hits, wcs, ocrs = 0, [], []
    for date_str, label in TEST_DATES:
        ok, wc, ocr = test_url(c['lccn'], date_str)
        if ok:
            hits += 1
            wcs.append(wc); ocrs.append(ocr)
        time.sleep(RATE_LIMIT)
    avg_wc  = sum(wcs)/len(wcs) if wcs else 0
    avg_ocr = sum(ocrs)/len(ocrs) if ocrs else 0
    results.append({
        'lccn': c['lccn'], 'name': c['name'], 'state': c['state'],
        'hits': hits, 'total': len(TEST_DATES),
        'success_rate': hits/len(TEST_DATES),
        'avg_word_count': round(avg_wc), 'avg_ocr': round(avg_ocr,3),
    })
    status = '✅' if hits>=4 else ('△' if hits>=2 else '✗')
    print(f"  {status} {c['name'][:35]:35s} ({c['state'][:12]:12s}) "
          f"| {hits}/{len(TEST_DATES)} | OCR={avg_ocr:.2f}")

result_df = pd.DataFrame(results).sort_values('success_rate', ascending=False)

테스트할 신문사 총: 6개

신문사당 5개 날짜 테스트 시작...

  ✗ The Sun                             (New York    ) | 0/5 | OCR=0.00
  ✗ Deseret Evening News                (Utah        ) | 0/5 | OCR=0.00
  ✗ Montgomery Advertiser               (Alabama     ) | 0/5 | OCR=0.00
  ✗ Los Angeles Herald                  (California  ) | 0/5 | OCR=0.00
  ✗ Salt Lake Herald                    (Utah        ) | 0/5 | OCR=0.00
  ✗ Washington Evening Star             (D.C.        ) | 0/5 | OCR=0.00


## Cell 4 — 결과 정리 및 저장

In [5]:
print('=== 신문사 탐색 결과 (성공률 순) ===\n')
print(result_df.to_string(index=False))

good = result_df[result_df['success_rate'] >= 0.8]
print(f"\n\n성공률 80% 이상: {len(good)}개")
print(good[['name','state','hits','avg_ocr']].to_string(index=False))

print('\n=== 주별 사용 가능 신문사 ===')
for state in ['West Virginia','Illinois','New York']:
    subset = good[good['state']==state]
    print(f"  {state}: {len(subset)}개")
    for _,r in subset.iterrows():
        print(f"    - {r['name']} ({r['lccn']})")

BASE = Path('/content/drive/MyDrive/경북대/인과추론')
out = BASE / '02_splits' / 'newspaper_discovery_v4.csv'
result_df.to_csv(out, index=False)
print(f'\n저장: {out}')

=== 신문사 탐색 결과 (성공률 순) ===

      lccn                    name      state  hits  total  success_rate  avg_word_count  avg_ocr
sn83030272                 The Sun   New York     0      5           0.0               0        0
sn83045555    Deseret Evening News       Utah     0      5           0.0               0        0
sn84020645   Montgomery Advertiser    Alabama     0      5           0.0               0        0
sn85042462      Los Angeles Herald California     0      5           0.0               0        0
sn85058130        Salt Lake Herald       Utah     0      5           0.0               0        0
sn83045604 Washington Evening Star       D.C.     0      5           0.0               0        0


성공률 80% 이상: 0개
Empty DataFrame
Columns: [name, state, hits, avg_ocr]
Index: []

=== 주별 사용 가능 신문사 ===
  West Virginia: 0개
  Illinois: 0개
  New York: 0개

저장: /content/drive/MyDrive/경북대/인과추론/02_splits/newspaper_discovery_v4.csv
